## Variáveis categóricas e one-hot encoding (conteúdo movido)

Algumas colunas da base são **categorias** (nomes de motivo, meio de transporte, UF, órgão etc.) e não possuem ordem natural.
Para usar essas informações em modelos de **regressão**, precisamos transformar categorias em números sem criar uma “ordem falsa”.

### Ideia do one-hot encoding (explicação simples)
Para uma categoria com `k` valores diferentes, o one-hot cria colunas do tipo **“é esta categoria?”** (0 ou 1).
Quando usamos `drop_first`, uma categoria fica como **referência** e a quantidade de colunas geradas reduz.

### Regra prática (cardinalidade)
- **Poucas categorias** (dezenas): one-hot costuma ser viável.
- **Muitas categorias** (milhares: nomes de servidor/município): one-hot puro gera muitas colunas quase todas com 0.

A célula abaixo calcula a **cardinalidade** (quantas categorias existem por coluna) e sugere onde one-hot é razoável.


In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

# CSV do conjunto SCDP — no repo costuma chamar-se DiariasEPassagens_ultimos_2_anos.csv; no notebook rene_estevam_deckers usa-se base_rene_estevam_deckers.csv (mesmo schema)
df = pd.read_csv('DiariasEPassagens_ultimos_2_anos.csv')

# Rode depois da carga no daily_rates *ou* no rene_estevam_deckers — `df` e `valor_total_num` já têm de existir.
# Se abrir só este notebook, carregue o mesmo CSV que o relatório (DiariasEPassagens_... ou base_rene_...) e rode as células de conversão como lá.

LIMITE_OK = 40  # até aqui one-hot "ingênuo" ainda é manejável em sklearn
LIMITE_ATENCAO = 120  # zona cinzenta: ou agrupa raros ou aceita matriz larga

linhas = []
for col in df.columns:
    if col == "valor_total_num":
        continue  # alvo numérico — não entra como categórica

    s = df[col]
    if pd.api.types.is_datetime64_any_dtype(s):
        continue  # datas tratamos em outro passo (mês, ano, etc.)

    nu = s.nunique(dropna=True)
    if nu <= 1:
        continue  # constante — não gera dummy útil

    is_num = pd.api.types.is_numeric_dtype(s)
    if is_num and nu > 200:
        continue  # trato como contínua/discreta com muitos valores, não como factor

    is_cat_like = (
        pd.api.types.is_object_dtype(s)
        or pd.api.types.is_string_dtype(s)
        or (is_num and nu <= 200)
    )
    if not is_cat_like:
        continue

    miss_pct = (s.isna().mean() * 100).round(2)
    colunas_sim_nao = max(0, int(nu) - 1)  # com drop_first=True são k-1 colunas dummy (referência cai fora)

    if nu <= LIMITE_OK:
        rec = "Poucas categorias: one-hot costuma ser viável."
    elif nu <= LIMITE_ATENCAO:
        rec = "Quantidade média: pode usar, mas vale agrupar categorias raras."
    else:
        rec = "Muitas categorias: evitar one-hot puro; agrupar ou subir nível."

    linhas.append(
        {
            "Coluna": col,
            "Tipo": str(s.dtype),
            "Categorias distintas": int(nu),
            "Colunas sim/não (estimativa, drop_first)": colunas_sim_nao,
            "% ausente": miss_pct,
            "Leitura prática": rec,
        }
    )

tab_ohe = pd.DataFrame(linhas).sort_values("Categorias distintas", ascending=False)
display(tab_ohe)

if len(tab_ohe):
    n_ok = int((tab_ohe["Categorias distintas"] <= LIMITE_OK).sum())
    n_meio = int(((tab_ohe["Categorias distintas"] > LIMITE_OK) & (tab_ohe["Categorias distintas"] <= LIMITE_ATENCAO)).sum())
    n_alto = int((tab_ohe["Categorias distintas"] > LIMITE_ATENCAO).sum())

    soma_ok = int(tab_ohe.loc[tab_ohe["Categorias distintas"] <= LIMITE_OK, "Colunas sim/não (estimativa, drop_first)"].sum())

    print("\nResumo: critérios de cardinalidade para one-hot")
    print("- Até %d categorias: %d coluna(s); soma estimada de colunas 0/1: %d" % (LIMITE_OK, n_ok, soma_ok))
    print("- Entre %d e %d categorias: %d coluna(s) (avaliar agrupar raros)." % (LIMITE_OK + 1, LIMITE_ATENCAO, n_meio))
    print("- Acima de %d categorias: %d coluna(s) (evitar one-hot puro)." % (LIMITE_ATENCAO, n_alto))

    # Ilustração rápida do formato real com pd.get_dummies (amostra)
    N_AMOSTRA = 5000
    amostra_demo = df.sample(n=min(N_AMOSTRA, len(df)), random_state=42)
    cols_demo = tab_ohe.loc[tab_ohe["Categorias distintas"] <= LIMITE_OK, "Coluna"].head(2).tolist()

    if len(cols_demo) < 2:
        candidatos = ["Meio de transporte", "Categoria passagem", "Motivo", "UF destino", "UF origem"]
        cols_demo = [c for c in candidatos if c in amostra_demo.columns][:2]

    if cols_demo:
        dummies = pd.get_dummies(amostra_demo[cols_demo], drop_first=True, dtype=int)
        print("\nIlustração com pd.get_dummies (amostra):")
        print("Colunas usadas:", cols_demo)
        print("Shape do bloco one-hot:", dummies.shape, "(linhas x colunas)")
        display(dummies.iloc[:6, : min(12, dummies.shape[1])])



C:\Users\Primodeckers\AppData\Local\Temp\ipykernel_7672\2644096480.py:5: DtypeWarning: Columns (0: Município origem, 1: UF origem, 2: Município destino, 3: UF destino) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('DiariasEPassagens_ultimos_2_anos.csv')


,Coluna,Tipo,Categorias distintas,"Colunas sim/não (estimativa, drop_first)",% ausente,Leitura prática
10,Valor total,str,228072,228071,0.00,Muitas categorias: evitar one-hot puro; agrupa...
5,Nome servidor,str,218237,218236,0.00,Muitas categorias: evitar one-hot puro; agrupa...
21,Valor passagem,str,95355,95354,0.00,Muitas categorias: evitar one-hot puro; agrupa...
9,Motivo,str,74340,74339,0.01,Muitas categorias: evitar one-hot puro; agrupa...
18,Valor diárias,str,23195,23194,0.00,Muitas categorias: evitar one-hot puro; agrupa...
4,Nome unidade gestora,str,3986,3985,0.00,Muitas categorias: evitar one-hot puro; agrupa...
15,Município destino,str,2325,2324,92.86,Muitas categorias: evitar one-hot puro; agrupa...
13,Município origem,str,1469,1468,92.68,Muitas categorias: evitar one-hot puro; agrupa...
6,Cargo,str,989,988,19.43,Muitas categorias: evitar one-hot puro; agrupa...
12,Término trecho,str,549,548,0.00,Muitas categorias: evitar one-hot puro; agrupa...



Resumo: critérios de cardinalidade para one-hot
- Até 40 categorias: 6 coluna(s); soma estimada de colunas 0/1: 133
- Entre 41 e 120 categorias: 0 coluna(s) (avaliar agrupar raros).
- Acima de 120 categorias: 16 coluna(s) (evitar one-hot puro).

Ilustração com pd.get_dummies (amostra):
Colunas usadas: ['UF destino', 'UF origem']
Shape do bloco one-hot: (5000, 44) (linhas x colunas)


,UF destino_AL,UF destino_AM,UF destino_BA,UF destino_CE,UF destino_DF,UF destino_ES,UF destino_GO,UF destino_MA,UF destino_MG,UF destino_MT,UF destino_PA,UF destino_PB
2236785,0,0,0,0,0,0,0,0,0,0,0,0
1766627,0,0,0,0,0,0,0,0,0,0,0,0
785496,0,0,0,0,0,0,0,0,0,0,0,0
1130950,0,0,0,0,0,0,0,0,0,0,0,0
371256,0,0,0,0,0,0,0,0,0,0,0,0
660056,0,0,0,0,0,0,0,0,0,0,0,0
